# SofaScore High-Rated Players by Team

This notebook identifies each team's best-performing players from the recent
**overall** and **Bundesliga** match windows stored in the newest timestamped
`team_form_*.json` snapshot. The two categories are evaluated independently.

## Runtime requirements and portability

- Place this notebook and at least one `team_form_*.json` file in the same
  project directory. The reference notebooks were used to design this workflow
  but are not runtime inputs.
- The project directory is resolved from the active notebook session where
  possible, with safe current-directory fallbacks. No fixed absolute project
  path or external configuration is required.
- The local Jupyter kernel must provide `undetected-chromedriver`, Selenium,
  and Beautiful Soup, plus a compatible local Chrome installation.
- One reusable browser is started with `uc.Chrome(version_main=150)` and the
  only SofaScore endpoint used is
  `https://www.sofascore.com/api/v1/event/{match_id}/lineups`.
- A player is evaluated when they have ratings in at least three matches, or
  in both of the two most recent available matches in that category. They are
  retained only when their average SofaScore rating is at least **7.00**.
- Missing ratings are ignored, never converted to zero. Overall and Bundesliga
  ratings are never mixed.
- Outputs are `team_high_rated_players_{timestamp}.json` and a matching `.csv`
  file in the project directory.

The latest input is selected from the timestamp encoded in each filename, not
alphabetical order or modification time. Live testing is performed by the user
locally; the generated notebook has been checked by Codex for Python syntax and
logical behavior without making SofaScore requests.


## 1. Imports and configuration

Configuration is kept inside the notebook. Dependency failures explain which
packages must be installed in the active Jupyter kernel.


In [ ]:
# Import the libraries required by this notebook step.
from __future__ import annotations

import csv
import json
import math
import os
import re
import ssl
import warnings
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from typing import Any
from urllib.parse import quote, urljoin
from urllib.request import Request, urlopen

# Handle expected failures with a clear, actionable message.
try:
    import undetected_chromedriver as uc
    from bs4 import BeautifulSoup
    from selenium.common.exceptions import TimeoutException, WebDriverException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait
except ImportError as exc:
    raise ImportError(
        "Required packages are missing from this Jupyter kernel. Install "
        "undetected-chromedriver, selenium, and beautifulsoup4, then restart "
        "the kernel."
    ) from exc

# Set workflow configuration value: CHROME_MAJOR_VERSION.
CHROME_MAJOR_VERSION = 150
# Set workflow configuration value: HEADLESS.
HEADLESS = False
# Set workflow configuration value: PAGE_LOAD_TIMEOUT_SECONDS.
PAGE_LOAD_TIMEOUT_SECONDS = 30
# Set workflow configuration value: WAIT_TIMEOUT_SECONDS.
WAIT_TIMEOUT_SECONDS = 20
# Set workflow configuration value: RATING_THRESHOLD.
RATING_THRESHOLD = Decimal("7.00")
# Set workflow configuration value: MINIMUM_RATED_MATCHES.
MINIMUM_RATED_MATCHES = 3
# Set workflow configuration value: LINEUPS_URL_TEMPLATE.
LINEUPS_URL_TEMPLATE = "https://www.sofascore.com/api/v1/event/{match_id}/lineups"

# Set workflow configuration value: TEAM_FORM_FILENAME_RE.
TEAM_FORM_FILENAME_RE = re.compile(
    r"^team_form_"
    r"(?P<date>\d{4}-\d{2}-\d{2})_"
    r"(?P<time>\d{2}-\d{2}-\d{2})"
    r"(?:_(?P<microseconds>\d{1,6}))?"
    r"(?P<offset>[+-]\d{4})?\.json$"
)

# Process each available item while preserving the current workflow state.
for setting_name, setting_value in {
    "CHROME_MAJOR_VERSION": CHROME_MAJOR_VERSION,
    "PAGE_LOAD_TIMEOUT_SECONDS": PAGE_LOAD_TIMEOUT_SECONDS,
    "WAIT_TIMEOUT_SECONDS": WAIT_TIMEOUT_SECONDS,
    "MINIMUM_RATED_MATCHES": MINIMUM_RATED_MATCHES,
}.items():
    # Validate the input before continuing with later processing.
    if (
        not isinstance(setting_value, int)
        or isinstance(setting_value, bool)
        or setting_value < 1
    ):
        raise ValueError(f"{setting_name} must be a positive integer.")


## 2. Locate the project directory

The resolver first uses frontend/session information to find the active
notebook. If that is unavailable, it checks the current directory, its
ancestors, and immediate child directories. It never embeds a machine-specific
project path.


In [ ]:
# Handle kernel ID for reuse in the workflow.
def _active_kernel_id() -> str | None:
    # Handle expected failures with a clear, actionable message.
    try:
        # Import the libraries required by this notebook step.
        from ipykernel.connect import get_connection_file

        connection_stem = Path(get_connection_file()).stem
        return connection_stem.split("-", 1)[-1]
    except Exception:
        return None


# Handle jupyter servers for reuse in the workflow.
def _running_jupyter_servers() -> list[dict[str, Any]]:
    providers = []
    # Handle expected failures with a clear, actionable message.
    try:
        from jupyter_server.serverapp import list_running_servers

        providers.append(list_running_servers)
    except Exception:
        pass
    # Handle expected failures with a clear, actionable message.
    try:
        from notebook.notebookapp import list_running_servers

        providers.append(list_running_servers)
    except Exception:
        pass

    servers: list[dict[str, Any]] = []
    seen_urls: set[str] = set()
    # Process each available item while preserving the current workflow state.
    for provider in providers:
        # Handle expected failures with a clear, actionable message.
        try:
            candidates = provider()
        except Exception:
            continue
        # Process each available item while preserving the current workflow state.
        for server in candidates:
            if not isinstance(server, dict):
                continue
            server_url = str(server.get("url") or "")
            if server_url and server_url not in seen_urls:
                seen_urls.add(server_url)
                servers.append(server)
    return servers


# Handle path from jupyter session for reuse in the workflow.
def _notebook_path_from_jupyter_session() -> Path | None:
    kernel_id = _active_kernel_id()
    if not kernel_id:
        return None

    # Process each available item while preserving the current workflow state.
    for server in _running_jupyter_servers():
        server_url = str(server.get("url") or "")
        if not server_url:
            continue
        sessions_url = urljoin(server_url, "api/sessions")
        token = str(server.get("token") or "")
        if token:
            separator = "&" if "?" in sessions_url else "?"
            sessions_url = f"{sessions_url}{separator}token={quote(token)}"

        request = Request(sessions_url)
        if token:
            request.add_header("Authorization", f"token {token}")
        context = (
            ssl._create_unverified_context()
            if sessions_url.lower().startswith("https://")
            else None
        )
        # Handle expected failures with a clear, actionable message.
        try:
            # Use the resource only within this controlled scope.
            with urlopen(request, timeout=2, context=context) as response:
                sessions = json.loads(response.read().decode("utf-8"))
        except Exception:
            continue
        if not isinstance(sessions, list):
            continue

        # Process each available item while preserving the current workflow state.
        for session in sessions:
            if not isinstance(session, dict):
                continue
            kernel = session.get("kernel")
            if not isinstance(kernel, dict) or kernel.get("id") != kernel_id:
                continue
            relative_path = session.get("path")
            if not relative_path:
                notebook_info = session.get("notebook")
                if isinstance(notebook_info, dict):
                    relative_path = notebook_info.get("path")
            if not relative_path:
                continue

            candidate = Path(str(relative_path))
            if candidate.is_absolute():
                return candidate.resolve()
            root = server.get("root_dir") or server.get("notebook_dir")
            if root:
                return (Path(str(root)) / candidate).resolve()
    return None


# Resolve project directory for reuse in the workflow.
def resolve_project_directory() -> Path:
    vscode_value = globals().get("__vsc_ipynb_file__")
    notebook_path = None
    if isinstance(vscode_value, (str, os.PathLike)) and str(vscode_value):
        notebook_path = Path(vscode_value).expanduser().resolve()
    if notebook_path is None:
        notebook_path = _notebook_path_from_jupyter_session()

    # Validate the input before continuing with later processing.
    if notebook_path is not None:
        notebook_directory = notebook_path.parent
        if list(notebook_directory.glob("team_form_*.json")):
            return notebook_directory
        raise FileNotFoundError(
            "The active notebook directory was resolved as "
            f"{notebook_directory}, but it contains no team_form_*.json file. "
            "Place the input beside this notebook."
        )

    current = Path.cwd().resolve()
    search_directories = [current, *current.parents]
    # Handle expected failures with a clear, actionable message.
    try:
        search_directories.extend(
            child for child in current.iterdir() if child.is_dir()
        )
    except OSError:
        pass

    matches: list[Path] = []
    seen: set[Path] = set()
    # Process each available item while preserving the current workflow state.
    for directory in search_directories:
        resolved = directory.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        # Handle expected failures with a clear, actionable message.
        try:
            contains_input = any(resolved.glob("team_form_*.json"))
        except OSError:
            contains_input = False
        if contains_input:
            matches.append(resolved)

    # Validate the input before continuing with later processing.
    if not matches:
        raise FileNotFoundError(
            "Could not resolve a project directory containing team_form_*.json. "
            "Place the input beside this notebook and launch the notebook from "
            "that project directory."
        )
    # Validate the input before continuing with later processing.
    if len(matches) > 1:
        choices = "\n".join(f"- {path}" for path in matches)
        raise RuntimeError(
            "Project-directory resolution is ambiguous. Matching directories:\n"
            f"{choices}"
        )
    return matches[0]


project_directory = resolve_project_directory()
print(f"Project directory:\n{project_directory}")


## 3. Find the latest `team_form_*.json`

Filename timestamps are parsed explicitly and compared as instants. A missing
timezone offset is interpreted using the machine's local timezone. Malformed
matching filenames are reported and ignored; an ambiguous latest instant is an
error.


In [ ]:
# Parse and validate team form filename timestamp for reuse in the workflow.
def parse_team_form_filename_timestamp(
    path: Path,
    local_timezone: Any | None = None,
) -> datetime:
    match = TEAM_FORM_FILENAME_RE.fullmatch(path.name)
    # Validate the input before continuing with later processing.
    if match is None:
        raise ValueError(
            f"Filename does not contain a supported team-form timestamp: {path.name}"
        )

    base = datetime.strptime(
        f"{match.group('date')}_{match.group('time')}",
        "%Y-%m-%d_%H-%M-%S",
    )
    microseconds_text = match.group("microseconds")
    if microseconds_text:
        base = base.replace(
            microsecond=int(microseconds_text.ljust(6, "0"))
        )

    offset_text = match.group("offset")
    # Validate the input before continuing with later processing.
    if offset_text:
        tzinfo = datetime.strptime(offset_text, "%z").tzinfo
    else:
        tzinfo = local_timezone or datetime.now().astimezone().tzinfo
        # Validate the input before continuing with later processing.
        if tzinfo is None:
            raise RuntimeError("Could not determine the machine's local timezone.")
    return base.replace(tzinfo=tzinfo).astimezone(timezone.utc)


# Select latest team form file for reuse in the workflow.
def select_latest_team_form_file(directory: Path) -> Path:
    candidates = sorted(directory.glob("team_form_*.json"))
    # Validate the input before continuing with later processing.
    if not candidates:
        raise FileNotFoundError(
            f"No team_form_*.json file could be found in {directory}."
        )

    parsed: list[tuple[datetime, Path]] = []
    malformed: list[tuple[Path, str]] = []
    local_timezone = datetime.now().astimezone().tzinfo
    # Process each available item while preserving the current workflow state.
    for path in candidates:
        # Handle expected failures with a clear, actionable message.
        try:
            timestamp = parse_team_form_filename_timestamp(
                path,
                local_timezone=local_timezone,
            )
        except ValueError as exc:
            malformed.append((path, str(exc)))
            continue
        parsed.append((timestamp, path))

    # Process each available item while preserving the current workflow state.
    for path, reason in malformed:
        warnings.warn(f"Ignoring {path.name}: {reason}", stacklevel=2)
    # Validate the input before continuing with later processing.
    if not parsed:
        raise ValueError(
            "Files matching team_form_*.json were found, but none contained a "
            "valid timestamp in the filename."
        )

    latest_timestamp = max(timestamp for timestamp, _ in parsed)
    latest_paths = [
        path for timestamp, path in parsed if timestamp == latest_timestamp
    ]
    # Validate the input before continuing with later processing.
    if len(latest_paths) != 1:
        names = ", ".join(path.name for path in latest_paths)
        raise RuntimeError(
            "Multiple team-form files encode the same latest instant: " + names
        )
    return latest_paths[0]


selected_input_path = select_latest_team_form_file(project_directory)
print("Using team-form input:")
print(selected_input_path.name)


## 4. Load and validate the team-form snapshot

The reference format has one ISO timestamp key containing a dictionary keyed by
team ID. A direct team-ID dictionary is also accepted. Invalid IDs, match-side
mappings, chronology fields, or duplicates within one match window are rejected
before Chrome starts.


In [ ]:
# Define Team Form Input Error to keep related behaviour explicit.
class TeamFormInputError(ValueError):
    '''Raised when the selected team-form snapshot is unusable.'''


# Check whether finite number for reuse in the workflow.
def is_finite_number(value: Any) -> bool:
    return (
        isinstance(value, (int, float))
        and not isinstance(value, bool)
        and math.isfinite(float(value))
    )


# Parse and validate iso datetime for reuse in the workflow.
def parse_iso_datetime(value: Any) -> datetime:
    # Validate the input before continuing with later processing.
    if not isinstance(value, str) or not value.strip():
        raise ValueError("date must be a non-empty ISO datetime string")
    normalized = value.strip()
    if normalized.endswith("Z"):
        normalized = normalized[:-1] + "+00:00"
    parsed = datetime.fromisoformat(normalized)
    # Validate the input before continuing with later processing.
    if parsed.tzinfo is None:
        local_timezone = datetime.now().astimezone().tzinfo
        # Validate the input before continuing with later processing.
        if local_timezone is None:
            raise ValueError("could not resolve timezone for a naive date")
        parsed = parsed.replace(tzinfo=local_timezone)
    return parsed.astimezone(timezone.utc)


# Handle chronology value for reuse in the workflow.
def match_chronology_value(match: dict[str, Any]) -> datetime:
    timestamp = match.get("timestamp")
    if is_finite_number(timestamp):
        # Handle expected failures with a clear, actionable message.
        try:
            return datetime.fromtimestamp(float(timestamp), tz=timezone.utc)
        except (OverflowError, OSError, ValueError):
            pass
    # Handle expected failures with a clear, actionable message.
    try:
        return parse_iso_datetime(match.get("date"))
    except (TypeError, ValueError) as exc:
        raise TeamFormInputError(
            f"Match {match.get('match_id')!r} has neither a usable timestamp "
            f"nor a usable date: {exc}"
        ) from exc


# Handle matches chronologically for reuse in the workflow.
def sort_matches_chronologically(
    matches: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    return sorted(
        matches,
        key=lambda match: (
            match_chronology_value(match),
            match["match_id"],
        ),
    )


# Validate match record for reuse in the workflow.
def _validate_match_record(
    raw_match: Any,
    team_id: int,
    category: str,
) -> dict[str, Any]:
    # Validate the input before continuing with later processing.
    if not isinstance(raw_match, dict):
        raise TeamFormInputError(
            f"Team {team_id} {category} contains a non-object match entry."
        )
    match = dict(raw_match)
    match_id = match.get("match_id")
    # Validate the input before continuing with later processing.
    if (
        not isinstance(match_id, int)
        or isinstance(match_id, bool)
        or match_id < 1
    ):
        raise TeamFormInputError(
            f"Team {team_id} {category} contains an invalid match_id."
        )

    home_team_id = match.get("home_team_id")
    away_team_id = match.get("away_team_id")
    # Process each available item while preserving the current workflow state.
    for field_name, value in (
        ("home_team_id", home_team_id),
        ("away_team_id", away_team_id),
    ):
        # Validate the input before continuing with later processing.
        if (
            not isinstance(value, int)
            or isinstance(value, bool)
            or value < 1
        ):
            raise TeamFormInputError(
                f"Match {match_id} has an invalid {field_name}."
            )
    # Validate the input before continuing with later processing.
    if team_id not in (home_team_id, away_team_id):
        raise TeamFormInputError(
            f"Match {match_id} lists neither side as target team {team_id}."
        )
    match_chronology_value(match)
    return match


# Load team form snapshot for reuse in the workflow.
def load_team_form_snapshot(path: Path) -> dict[int, dict[str, Any]]:
    # Handle expected failures with a clear, actionable message.
    try:
        raw = json.loads(path.read_text(encoding="utf-8"))
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"Team-form input was not found: {path}") from exc
    except UnicodeDecodeError as exc:
        raise TeamFormInputError(f"Input is not valid UTF-8: {path}") from exc
    except json.JSONDecodeError as exc:
        raise TeamFormInputError(
            f"Input is not valid JSON (line {exc.lineno}, column {exc.colno}): "
            f"{path}"
        ) from exc
    except OSError as exc:
        raise OSError(f"Could not read {path}: {exc}") from exc

    # Validate the input before continuing with later processing.
    if not isinstance(raw, dict) or not raw:
        raise TeamFormInputError("The team-form JSON must be a non-empty object.")

    top_level_keys_are_team_ids = all(
        str(key).isdigit() for key in raw.keys()
    )
    # Validate the input before continuing with later processing.
    if top_level_keys_are_team_ids:
        raw_teams = raw
    # Validate the input before continuing with later processing.
    elif len(raw) == 1:
        snapshot_key, raw_teams = next(iter(raw.items()))
        # Handle expected failures with a clear, actionable message.
        try:
            parse_iso_datetime(snapshot_key)
        except (TypeError, ValueError) as exc:
            raise TeamFormInputError(
                f"Snapshot key is not a valid ISO datetime: {snapshot_key!r}"
            ) from exc
        # Validate the input before continuing with later processing.
        if not isinstance(raw_teams, dict) or not raw_teams:
            raise TeamFormInputError(
                "The timestamped snapshot must contain a non-empty team object."
            )
    else:
        raise TeamFormInputError(
            "Expected either a team-ID object or one timestamp key containing "
            "the team-ID object."
        )

    teams: dict[int, dict[str, Any]] = {}
    # Process each available item while preserving the current workflow state.
    for team_key, raw_team in raw_teams.items():
        # Handle expected failures with a clear, actionable message.
        try:
            team_id = int(str(team_key))
        except (TypeError, ValueError) as exc:
            raise TeamFormInputError(
                f"Invalid team key: {team_key!r}"
            ) from exc
        # Validate the input before continuing with later processing.
        if team_id < 1 or str(team_id) != str(team_key):
            raise TeamFormInputError(f"Invalid numeric team key: {team_key!r}")
        # Validate the input before continuing with later processing.
        if not isinstance(raw_team, dict):
            raise TeamFormInputError(f"Team {team_id} must be a JSON object.")
        embedded_team_id = raw_team.get("team_id")
        # Validate the input before continuing with later processing.
        if embedded_team_id is not None and embedded_team_id != team_id:
            raise TeamFormInputError(
                f"Team key {team_id} does not match embedded team_id "
                f"{embedded_team_id!r}."
            )
        team_name = raw_team.get("team")
        # Validate the input before continuing with later processing.
        if not isinstance(team_name, str) or not team_name.strip():
            raise TeamFormInputError(f"Team {team_id} has no valid team name.")

        validated_team: dict[str, Any] = {"team": team_name.strip()}
        # Process each available item while preserving the current workflow state.
        for category, field_name in (
            ("overall", "overall_matches"),
            ("bundesliga", "bundesliga_matches"),
        ):
            raw_matches = raw_team.get(field_name)
            # Validate the input before continuing with later processing.
            if not isinstance(raw_matches, list):
                raise TeamFormInputError(
                    f"Team {team_id} field {field_name} must be a list."
                )
            matches = [
                _validate_match_record(match, team_id, category)
                for match in raw_matches
            ]
            match_ids = [match["match_id"] for match in matches]
            # Validate the input before continuing with later processing.
            if len(match_ids) != len(set(match_ids)):
                raise TeamFormInputError(
                    f"Team {team_id} {category} contains duplicate match IDs."
                )
            validated_team[field_name] = matches
        teams[team_id] = validated_team

    return teams


teams = load_team_form_snapshot(selected_input_path)
print(f"Loaded and validated {len(teams)} team(s).")


## 5. Start one reusable undetected Chrome session

Browser creation is encapsulated so the final pipeline can guarantee cleanup.
Headless mode is disabled by default for compatibility with the reference
extraction workflow.


In [ ]:
# Create browser for reuse in the workflow.
def create_browser() -> Any:
    options = uc.ChromeOptions()
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    if HEADLESS:
        options.add_argument("--headless=new")

    driver = None
    # Handle expected failures with a clear, actionable message.
    try:
        driver = uc.Chrome(
            options=options,
            version_main=150,
            use_subprocess=True,
        )
        driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SECONDS)
        print(
            "One reusable undetected Chrome session is ready "
            f"(major version {CHROME_MAJOR_VERSION}, headless={HEADLESS})."
        )
        return driver
    except Exception as exc:
        if driver is not None:
            # Handle expected failures with a clear, actionable message.
            try:
                driver.quit()
            except Exception:
                pass
        raise RuntimeError(
            "Could not initialize undetected Chrome with major version 150. "
            f"Original error: {type(exc).__name__}: {exc}"
        ) from exc


# Handle browser for reuse in the workflow.
def close_browser(driver: Any | None) -> None:
    if driver is None:
        print("Chrome was not initialized.")
        return
    # Handle expected failures with a clear, actionable message.
    try:
        driver.quit()
        print("Chrome driver closed.")
    except Exception as exc:
        print(f"Chrome driver shutdown warning: {type(exc).__name__}: {exc}")


## 6. SofaScore retrieval and match-response cache

Responses are read from the rendered `<pre>` element. Both successful payloads
and failures are cached, which guarantees that a shared match ID is never
requested twice during the run.


In [ ]:
# Define Sofa Score Lineup Error to keep related behaviour explicit.
class SofaScoreLineupError(RuntimeError):
    '''Raised when a lineup endpoint cannot return a usable payload.'''


# Retrieve lineup payload for reuse in the workflow.
def fetch_lineup_payload(driver: Any, match_id: int) -> dict[str, Any]:
    url = LINEUPS_URL_TEMPLATE.format(match_id=match_id)
    # Handle expected failures with a clear, actionable message.
    try:
        driver.get(url)
    except TimeoutException as exc:
        raise SofaScoreLineupError(
            f"Timed out after {PAGE_LOAD_TIMEOUT_SECONDS} seconds loading {url}."
        ) from exc
    except WebDriverException as exc:
        raise SofaScoreLineupError(f"Chrome could not load {url}: {exc}") from exc

    # Handle expected failures with a clear, actionable message.
    try:
        WebDriverWait(driver, WAIT_TIMEOUT_SECONDS).until(
            EC.presence_of_element_located((By.TAG_NAME, "pre"))
        )
    except TimeoutException as exc:
        raise SofaScoreLineupError(
            f"No <pre> element appeared within {WAIT_TIMEOUT_SECONDS} seconds "
            f"for {url}."
        ) from exc
    except WebDriverException as exc:
        raise SofaScoreLineupError(
            f"Chrome could not inspect the rendered response for {url}: {exc}"
        ) from exc

    soup = BeautifulSoup(driver.page_source, "html.parser")
    pre_tag = soup.find("pre")
    # Validate the input before continuing with later processing.
    if pre_tag is None:
        raise SofaScoreLineupError(f"Rendered page contains no <pre>: {url}")
    response_text = pre_tag.get_text().strip()
    # Validate the input before continuing with later processing.
    if not response_text:
        raise SofaScoreLineupError(f"The <pre> element is empty: {url}")
    # Handle expected failures with a clear, actionable message.
    try:
        payload = json.loads(response_text)
    except json.JSONDecodeError as exc:
        raise SofaScoreLineupError(
            f"Invalid JSON for {url} (line {exc.lineno}, column {exc.colno})."
        ) from exc
    # Validate the input before continuing with later processing.
    if not isinstance(payload, dict):
        raise SofaScoreLineupError(f"Lineup response is not an object: {url}")
    # Process each available item while preserving the current workflow state.
    for side in ("home", "away"):
        side_payload = payload.get(side)
        # Validate the input before continuing with later processing.
        if not isinstance(side_payload, dict) or not isinstance(
            side_payload.get("players"), list
        ):
            raise SofaScoreLineupError(
                f"Lineup response has no usable {side}.players list: {url}"
            )
    return payload


# Handle cached lineup for reuse in the workflow.
def get_cached_lineup(
    driver: Any,
    match_id: int,
    match_cache: dict[int, dict[str, Any]],
    request_counters: dict[str, int],
) -> dict[str, Any]:
    if match_id in match_cache:
        request_counters["cache_hits"] += 1
        return match_cache[match_id]

    # Handle expected failures with a clear, actionable message.
    try:
        payload = fetch_lineup_payload(driver, match_id)
    except Exception as exc:
        result = {
            "ok": False,
            "payload": None,
            "error": f"{type(exc).__name__}: {exc}",
        }
        request_counters["failed_requests"] += 1
    else:
        result = {"ok": True, "payload": payload, "error": None}
        request_counters["successful_requests"] += 1
    match_cache[match_id] = result
    return result


## 7. Target-team player-rating extraction

The home/away side is determined only from the supplied match record and target
team ID. Opponent players are never included. Player identity is based on
`player.id`; entries without a finite `statistics.rating` are ignored.


In [ ]:
# Handle team side for reuse in the workflow.
def determine_team_side(match: dict[str, Any], team_id: int) -> str:
    if match.get("home_team_id") == team_id:
        return "home"
    if match.get("away_team_id") == team_id:
        return "away"
    raise TeamFormInputError(
        f"Match {match.get('match_id')} does not contain team {team_id}."
    )


# Extract rated players for side for reuse in the workflow.
def extract_rated_players_for_side(
    payload: dict[str, Any],
    side: str,
    match_id: int,
) -> list[dict[str, Any]]:
    side_payload = payload.get(side)
    players = side_payload.get("players") if isinstance(side_payload, dict) else None
    # Validate the input before continuing with later processing.
    if not isinstance(players, list):
        raise SofaScoreLineupError(
            f"Match {match_id} has no usable {side}.players list."
        )

    extracted: list[dict[str, Any]] = []
    seen_player_ids: set[int] = set()
    # Process each available item while preserving the current workflow state.
    for entry in players:
        if not isinstance(entry, dict):
            continue
        player = entry.get("player")
        statistics = entry.get("statistics")
        if not isinstance(player, dict) or not isinstance(statistics, dict):
            continue
        player_id = player.get("id")
        rating = statistics.get("rating")
        if (
            not isinstance(player_id, int)
            or isinstance(player_id, bool)
            or player_id < 1
            or not is_finite_number(rating)
        ):
            continue
        # Validate the input before continuing with later processing.
        if player_id in seen_player_ids:
            raise SofaScoreLineupError(
                f"Match {match_id} contains duplicate rated entries for "
                f"player_id {player_id} on the {side} side."
            )
        seen_player_ids.add(player_id)

        player_name = player.get("name")
        # Choose the appropriate path for the current data state.
        if not isinstance(player_name, str) or not player_name.strip():
            player_name = None
        else:
            player_name = player_name.strip()
        position = entry.get("position")
        # Choose the appropriate path for the current data state.
        if not isinstance(position, str) or not position.strip():
            position = None
        else:
            position = position.strip()

        extracted.append(
            {
                "player_id": player_id,
                "player_name": player_name,
                "position": position,
                "rating": float(rating),
                "match_id": match_id,
            }
        )
    return extracted


## 8. Match ordering and player qualification

Each category is sorted oldest-to-newest. Qualification uses either three rated
matches or both specifically newest matches, followed by the unrounded average
threshold. Output averages use decimal half-up rounding to two places.


In [ ]:
# Handle match context for reuse in the workflow.
def failed_match_context(
    match_id: int,
    team_id: int,
    team_name: str,
    category: str,
    reason: str,
) -> dict[str, Any]:
    return {
        "match_id": match_id,
        "team_id": team_id,
        "team": team_name,
        "category": category,
        "error": reason,
    }


# Handle category for reuse in the workflow.
def evaluate_category(
    driver: Any,
    team_id: int,
    team_name: str,
    category: str,
    matches: list[dict[str, Any]],
    match_cache: dict[int, dict[str, Any]],
    request_counters: dict[str, int],
    failed_matches: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    ordered_matches = sort_matches_chronologically(matches)
    last_two_ids = (
        {match["match_id"] for match in ordered_matches[-2:]}
        if len(ordered_matches) >= 2
        else set()
    )
    player_buckets: dict[int, dict[str, Any]] = {}

    # Process each available item while preserving the current workflow state.
    for match in ordered_matches:
        match_id = match["match_id"]
        cached = get_cached_lineup(
            driver,
            match_id,
            match_cache,
            request_counters,
        )
        if not cached["ok"]:
            failed_matches.append(
                failed_match_context(
                    match_id,
                    team_id,
                    team_name,
                    category,
                    str(cached["error"]),
                )
            )
            continue

        side = determine_team_side(match, team_id)
        # Handle expected failures with a clear, actionable message.
        try:
            rated_players = extract_rated_players_for_side(
                cached["payload"], side, match_id
            )
        except Exception as exc:
            failed_matches.append(
                failed_match_context(
                    match_id,
                    team_id,
                    team_name,
                    category,
                    f"{type(exc).__name__}: {exc}",
                )
            )
            continue

        # Process each available item while preserving the current workflow state.
        for rated in rated_players:
            player_id = rated["player_id"]
            bucket = player_buckets.setdefault(
                player_id,
                {
                    "player_name": None,
                    "position": None,
                    "ratings": [],
                    "rated_match_ids": set(),
                },
            )
            if rated["player_name"]:
                bucket["player_name"] = rated["player_name"]
            if rated["position"]:
                bucket["position"] = rated["position"]
            bucket["ratings"].append(
                {
                    "match_id": match_id,
                    "rating": rated["rating"],
                }
            )
            bucket["rated_match_ids"].add(match_id)

    qualified_with_means: list[tuple[Decimal, dict[str, Any]]] = []
    # Process each available item while preserving the current workflow state.
    for player_id, bucket in player_buckets.items():
        rating_records = bucket["ratings"]
        rating_count = len(rating_records)
        played_last_two = bool(last_two_ids) and last_two_ids.issubset(
            bucket["rated_match_ids"]
        )
        if not (
            rating_count >= MINIMUM_RATED_MATCHES or played_last_two
        ):
            continue

        decimal_ratings = [
            Decimal(str(record["rating"])) for record in rating_records
        ]
        average = sum(decimal_ratings, Decimal("0")) / Decimal(rating_count)
        if average < RATING_THRESHOLD:
            continue
        rounded_average = average.quantize(
            Decimal("0.01"), rounding=ROUND_HALF_UP
        )
        result = {
            "player_id": player_id,
            "player_name": bucket["player_name"],
            "position": bucket["position"],
            "average_rating": float(rounded_average),
            "ratings": rating_records,
        }
        qualified_with_means.append((average, result))

    qualified_with_means.sort(
        key=lambda item: (
            -item[0],
            (item[1]["player_name"] or "").casefold(),
            item[1]["player_id"],
        )
    )
    return [result for _, result in qualified_with_means]


## 9. Overall-form and Bundesliga-form processing

Every team is evaluated twice with separate match windows and separate player
aggregates. The shared match cache is the only state reused between categories.


In [ ]:
# Handle all teams for reuse in the workflow.
def process_all_teams(
    driver: Any,
    teams: dict[int, dict[str, Any]],
    match_cache: dict[int, dict[str, Any]],
    request_counters: dict[str, int],
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    failed_matches: list[dict[str, Any]] = []
    team_results: dict[str, Any] = {}

    # Process each available item while preserving the current workflow state.
    for team_number, (team_id, team_record) in enumerate(
        teams.items(), start=1
    ):
        team_name = team_record["team"]
        print(
            f"[{team_number}/{len(teams)}] {team_name} "
            f"(team_id={team_id})"
        )
        overall_players = evaluate_category(
            driver=driver,
            team_id=team_id,
            team_name=team_name,
            category="overall",
            matches=team_record["overall_matches"],
            match_cache=match_cache,
            request_counters=request_counters,
            failed_matches=failed_matches,
        )
        bundesliga_players = evaluate_category(
            driver=driver,
            team_id=team_id,
            team_name=team_name,
            category="bundesliga",
            matches=team_record["bundesliga_matches"],
            match_cache=match_cache,
            request_counters=request_counters,
            failed_matches=failed_matches,
        )
        team_results[str(team_id)] = {
            "team": team_name,
            "overall": {"players": overall_players},
            "bundesliga": {"players": bundesliga_players},
        }
    return team_results, failed_matches


# Handle input match IDs for reuse in the workflow.
def unique_input_match_ids(
    teams: dict[int, dict[str, Any]],
) -> set[int]:
    return {
        match["match_id"]
        for team_record in teams.values()
        for field_name in ("overall_matches", "bundesliga_matches")
        for match in team_record[field_name]
    }


## 10. Export JSON and CSV

One timezone-aware run timestamp is reused across the metadata and both
Windows-safe filenames. JSON retains the individual rating trail. CSV contains
exactly one row per qualifying player and category.


In [ ]:
# Set workflow configuration value: CSV_COLUMNS.
CSV_COLUMNS = [
    "team_id",
    "team",
    "category",
    "player_id",
    "player_name",
    "position",
    "average_rating",
]


# Handle results for reuse in the workflow.
def export_results(
    project_directory: Path,
    source_path: Path,
    generated_datetime: datetime,
    team_results: dict[str, Any],
    failed_matches: list[dict[str, Any]],
) -> tuple[dict[str, Any], Path, Path]:
    generated_at = generated_datetime.isoformat(timespec="seconds")
    filename_timestamp = generated_datetime.strftime(
        "%Y-%m-%d_%H-%M-%S_%f%z"
    )
    json_path = (
        project_directory
        / f"team_high_rated_players_{filename_timestamp}.json"
    )
    csv_path = (
        project_directory
        / f"team_high_rated_players_{filename_timestamp}.csv"
    )

    output_document = {
        "generated_at": generated_at,
        "source_file": source_path.name,
        "rating_threshold": float(RATING_THRESHOLD),
        "qualification": {
            "minimum_rated_matches": MINIMUM_RATED_MATCHES,
            "last_two_exception": True,
        },
        "teams": team_results,
        "failed_matches": failed_matches,
    }
    # Handle expected failures with a clear, actionable message.
    try:
        json_path.write_text(
            json.dumps(output_document, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
    except OSError as exc:
        raise OSError(f"Could not write JSON output {json_path}: {exc}") from exc

    # Handle expected failures with a clear, actionable message.
    try:
        # Use the resource only within this controlled scope.
        with csv_path.open("w", encoding="utf-8-sig", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=CSV_COLUMNS)
            writer.writeheader()
            # Process each available item while preserving the current workflow state.
            for team_id, team_result in team_results.items():
                # Process each available item while preserving the current workflow state.
                for category in ("overall", "bundesliga"):
                    # Process each available item while preserving the current workflow state.
                    for player in team_result[category]["players"]:
                        writer.writerow(
                            {
                                "team_id": team_id,
                                "team": team_result["team"],
                                "category": category,
                                "player_id": player["player_id"],
                                "player_name": player["player_name"],
                                "position": player["position"],
                                "average_rating": (
                                    f"{player['average_rating']:.2f}"
                                ),
                            }
                        )
    except OSError as exc:
        raise OSError(f"Could not write CSV output {csv_path}: {exc}") from exc

    return output_document, json_path, csv_path


## 11. Processing summary

The summary distinguishes unique network attempts from cache reuse and reports
qualifying-player totals for each independent category.


In [ ]:
# Handle processing summary for reuse in the workflow.
def print_processing_summary(
    source_path: Path,
    teams: dict[int, dict[str, Any]],
    team_results: dict[str, Any],
    request_counters: dict[str, int],
    json_path: Path,
    csv_path: Path,
) -> None:
    overall_count = sum(
        len(team["overall"]["players"])
        for team in team_results.values()
    )
    bundesliga_count = sum(
        len(team["bundesliga"]["players"])
        for team in team_results.values()
    )
    print("\nProcessing summary")
    print("------------------")
    print(f"Selected input file: {source_path.name}")
    print(f"Teams processed: {len(team_results)} / {len(teams)}")
    print(f"Unique match IDs encountered: {len(unique_input_match_ids(teams))}")
    print(
        "Successful SofaScore requests: "
        f"{request_counters['successful_requests']}"
    )
    print(
        "Failed SofaScore requests: "
        f"{request_counters['failed_requests']}"
    )
    print(f"Cache hits / reused matches: {request_counters['cache_hits']}")
    print(f"Qualifying overall players: {overall_count}")
    print(f"Qualifying Bundesliga players: {bundesliga_count}")
    print(f"JSON output: {json_path}")
    print(f"CSV output: {csv_path}")


## 12. Run the pipeline and close Chrome

Run this final cell to retrieve the supplied match IDs, process both categories,
write the outputs, and print the summary. The `finally` block closes Chrome even
if processing or export encounters an unexpected error.


In [ ]:
driver = None
match_cache: dict[int, dict[str, Any]] = {}
request_counters = {
    "successful_requests": 0,
    "failed_requests": 0,
    "cache_hits": 0,
}
final_output_document = None
json_output_path = None
csv_output_path = None

run_datetime = datetime.now().astimezone()
# Handle expected failures with a clear, actionable message.
try:
    driver = create_browser()
    final_team_results, failed_matches = process_all_teams(
        driver=driver,
        teams=teams,
        match_cache=match_cache,
        request_counters=request_counters,
    )
    (
        final_output_document,
        json_output_path,
        csv_output_path,
    ) = export_results(
        project_directory=project_directory,
        source_path=selected_input_path,
        generated_datetime=run_datetime,
        team_results=final_team_results,
        failed_matches=failed_matches,
    )
    print_processing_summary(
        source_path=selected_input_path,
        teams=teams,
        team_results=final_team_results,
        request_counters=request_counters,
        json_path=json_output_path,
        csv_path=csv_output_path,
    )
finally:
    close_browser(driver)
    driver = None
